# STREME / TOMTOM / FIMO pipeline

This notebook configures, runs and summarizes the motif pipeline. The reusable
implementation is in `streme_pipeline.py`; the same code is used by the Slurm
array jobs.


## Setup


In [1]:
from pathlib import Path
import pandas as pd

from streme_pipeline import (
    PipelineConfig,
    PipelineParameters,
    STAGES,
    build_status,
    combine_result_tables,
    discover_fastas,
    export_fimo_for_binding_bench,
    run_all_stages,
    run_pipeline_for_fasta,
    save_analysis_tables,
    save_summaries,
)


In [2]:
config = PipelineConfig.default(
    project=Path("/s/project/ml4rg_students/2026/project15")
)
config.validate()

fastas = discover_fastas(config.fasta_dir)
parameters = PipelineParameters(
    streme_time=1800,
    minw=6,
    maxw=20,
    nmotifs=10,
    fimo_thresh="1e-4",
    fimo_max_stored_scores=100_000,
    fimo_skip_matched_sequence=False,
)

print(f"Found {len(fastas)} FASTA files")
print("FASTA directory:", config.fasta_dir)
print("Result directory:", config.result_dir)
print("JASPAR database:", config.jaspar_fungi)


Found 1402 FASTA files
FASTA directory: /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas
Result directory: /s/project/ml4rg_students/2026/project15/working/streme_results
JASPAR database: /s/project/ml4rg_students/2026/project15/working/jaspar/JASPAR2026_CORE_fungi_non-redundant_pfms_meme.txt


The pipeline writes a `.pipeline_done.json` next to each successful
result. New results are only reused when their command and input signatures
match. Existing results from the old notebook have no manifest and are accepted
as a legacy cache by default.


## Status


In [27]:
!squeue -j 19483294

             JOBID PARTITION     NAME     USER ST       TIME  NODES NODELIST(REASON)


In [28]:
!sacct -X -j 19483294 --format=JobID,JobName,State,ExitCode,Elapsed,MaxRSS

JobID           JobName      State ExitCode    Elapsed     MaxRSS 
------------ ---------- ---------- -------- ---------- ---------- 
19483294     bb-fimo-s+  COMPLETED      0:0   00:00:32            


In [5]:
status = build_status(config, fastas)
display(status.head())
display(status[list(STAGES)].sum().rename("completed"))

incomplete = status.loc[~status[list(STAGES)].all(axis=1)]
print(f"Incomplete datasets: {len(incomplete)}")
display(incomplete.head(20))


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
0,DNA_rossi_chipexo_sites,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False
1,_candida_arabinofermentans_nrrl_yb_2248_gca_00...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
2,_candida_auris_gca_001189475_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
3,_candida_auris_gca_002775015_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True
4,_candida_auris_gca_003013715_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,True


streme         497
fimo_jaspar     74
tomtom          62
fimo_streme     24
Name: completed, dtype: int64

Incomplete datasets: 1378


,name,fasta,streme,tomtom,fimo_jaspar,fimo_streme
0,DNA_rossi_chipexo_sites,/s/project/ml4rg_students/2026/project15/worki...,False,False,False,False
24,absidia_repens_gca_002105175_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
25,acaromyces_ingoldii_gca_003144295_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
26,acidomyces_richmondensis_bfw_gca_001592465_seq...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
27,acidomyces_sp_richmondensis_gca_001572075_sequ...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
28,acremonium_chrysogenum_atcc_11550_gca_00076926...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
29,agaricus_bisporus_var_burnettii_jb137_s8_gca_0...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
30,agrocybe_aegerita_gca_902728275_sequence_mapper,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
32,akanthomyces_lecanii_rcef_1005_gca_001636795_s...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False
33,allomyces_macrogynus_atcc_38327_gca_000151295_...,/s/project/ml4rg_students/2026/project15/worki...,True,True,True,False


In [24]:
from pathlib import Path
import pyarrow.parquet as pq
from pyfaidx import Fasta

parquet_file = Path("/s/project/multispecies/fungi_code/tf_sae/binding_bench_datasets/val/dna/_saccharomyces_cerevisiae/DNA_rossi_chipexo_sites.parquet")
genome_fasta = Path("/s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/DNA_rossi_chipexo_sites.fasta")

output_dir = Path("output")
output_dir.mkdir(parents=True, exist_ok=True)

out_path = output_dir / "DNA_rossi_chipexo_sites.fasta"

# Choose the flank size around the 1-bp ChIP-exo site
flank = 50   # gives 101 bp total if centered on a 1-bp site

cols = ["chrom", "start", "end", "strand", "site_id", "name"]
table = pq.read_table(parquet_file, columns=cols)
df = table.to_pandas()

genome = Fasta(str(genome_fasta))

def revcomp(seq):
    return seq.translate(str.maketrans("ACGTNacgtn", "TGCANtgcan"))[::-1]

def resolve_chrom(chrom, genome):
    """
    Handles FASTA headers like II vs chrII.
    """
    chrom = str(chrom)

    if chrom in genome:
        return chrom

    if f"chr{chrom}" in genome:
        return f"chr{chrom}"

    if chrom.startswith("chr") and chrom[3:] in genome:
        return chrom[3:]

    raise KeyError(
        f"Could not find chromosome {chrom!r} in FASTA. "
        f"Example FASTA keys: {list(genome.keys())[:10]}"
    )

with open(out_path, "w") as out:
    for _, row in df.iterrows():
        chrom_raw = row["chrom"]
        chrom = resolve_chrom(chrom_raw, genome)

        start = int(row["start"])
        end = int(row["end"])
        strand = row["strand"]
        site_id = row["site_id"]
        tf_name = row["name"]

        # Center window on the site midpoint
        center = (start + end) // 2
        window_start = max(0, center - flank)
        window_end = center + flank + 1

        seq = str(genome[chrom][window_start:window_end]).upper()

        # Your strand column is ".", so this will usually do nothing
        if strand == "-":
            seq = revcomp(seq)

        header = (
            f">{site_id}|tf={tf_name}|"
            f"{chrom_raw}:{window_start}-{window_end}|strand={strand}"
        )

        out.write(header + "\n")
        out.write(seq + "\n")

print(f"Wrote FASTA to: {out_path}")

FastaIndexingError: The FASTA file /s/project/ml4rg_students/2026/project15/working/sequence_datasets_fastas/DNA_rossi_chipexo_sites.fasta does not contain a valid sequence. Check that sequence definition lines start with '>'.

## Test one FASTA


In [7]:
from pathlib import Path

test_fasta = Path(
    "/s/project/ml4rg_students/2026/project15/working/"
    "sequence_datasets_fastas/_saccharomyces_cerevisiae_sequence_mapper.fasta"
)

test_result = run_pipeline_for_fasta(
    config,
    test_fasta,
    parameters=parameters,
    force=False,
    accept_legacy=True,
)

test_result


[_saccharomyces_cerevisiae_sequence_mapper] streme: existing legacy result accepted
[_saccharomyces_cerevisiae_sequence_mapper] fimo_jaspar: cached
[_saccharomyces_cerevisiae_sequence_mapper] tomtom: cached
[_saccharomyces_cerevisiae_sequence_mapper] fimo_streme: cached


{'streme': 'legacy',
 'fimo_jaspar': 'cached',
 'tomtom': 'cached',
 'fimo_streme': 'cached'}

In [9]:
prediction_path = export_fimo_for_binding_bench(
    config,
    test_fasta,
    result_key="fimo_streme_tsv",
)

prediction_path

PosixPath('/s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv')

In [10]:
!find /data/nasif12/home_if12/s_kmill -name pyproject.toml -path "*/binding_bench/*" 2>/dev/null

/data/nasif12/home_if12/s_kmill/binding_bench/pyproject.toml


In [11]:
!pip install -e /data/nasif12/home_if12/s_kmill/binding_bench

Obtaining file:///data/nasif12/home_if12/s_kmill/binding_bench
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Installing backend dependencies ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for binding_bench (pyproject.toml) ... done
  Created wheel for binding_bench: filename=binding_bench-0.0.1-py2.py3-none-any.whl size=1309 sha256=e1e8dc4fec1072b94c65b942c87a291b5c382eb4bde9ef4298b3f9e559ab772e
  Stored in directory: /scratch/tmp/s_kmill/pip-ephem-wheel-cache-w1u9ef68/wheels/a8/b4/e0/4b7cf52c0bdccd67020c28e7d5ca9092188f95c689f63e4d8e
Successfully built binding_bench
  Attempting uninstall: binding_bench
    Found existing installation: binding_bench 0.0.1
    Uninstalling binding_bench-0.0.1:
      Successfully uninstalled binding_bench-0.0.1


In [12]:
!python -m binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --display_name STREME-FIMO \
  --overwrite

In [13]:
!ls -lh /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv

-rw-rw----+ 1 s_kmill root 4.8M Jun 18 14:26 /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv


In [14]:
!head /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv

chrom	start	end	feature_idx	score	strand
XV	444507	444508	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-
X	136919	136920	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
XIII	851722	851723	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
IX	241041	241042	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
VIII	187038	187039	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
I	70915	70916	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
XVI	840644	840645	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-
II	145907	145908	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	+
VII	733121	733122	1-GAAAAAAAAAAAAAAAAAA	10.752026733638193	-


In [15]:
!find /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run -type f

In [16]:
!python -m binding_bench discrete --help

In [17]:
!ls -lh /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/

total 4.8M
-rw-rw----+ 1 s_kmill root 4.8M Jun 18 14:26 _saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv
drwxrws---+ 2 s_kmill root    0 Jun 18 13:57 streme_run
-rw-rw----+ 1 s_kmill root    0 Jun 18 14:02 streme_run.log


In [18]:
!mkdir -p /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run

!python -m binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --display_name STREME-FIMO \
  --overwrite

In [19]:
!find /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run -type f

In [20]:
!echo $?

0


In [21]:
!python -m binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/_saccharomyces_cerevisiae_sequence_mapper_fimo_streme_tsv.tsv \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --display_name STREME-FIMO \
  --overwrite \
  2>&1 | tee /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run.log

In [22]:
!echo $?
!cat /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run.log
!find /s/project/ml4rg_students/2026/project15/working/streme_results/binding_bench/streme_run -type f

0


In [29]:
from pathlib import Path
import pandas as pd

run_dir = Path(
    "/s/project/ml4rg_students/2026/project15/working/"
    "binding_bench_runs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper/"
    "binding_bench/discrete/DNA_rossi_chipexo"
)

full = pd.read_parquet(run_dir / "full_discrete_metrics_ds_DNA_rossi_chipexo.parquet")
full.head()

,feature_idx,name,TP_s,TP_p,n_overlap,sites_at_k,peaks_at_k,sites_total,peaks_total,precision,recall,jaccard,f1,precision_lb,precision_ub,recall_lb,recall_ub,sites_frac_bg,prec_lb_fc
0,4-YRYTAYYRHY,toa1,8,11,11,1517,1517,1517,2015,0.007251,0.005274,0.002644,0.006106,0.003625,0.012937,0.002279,0.010364,0.006101,-0.750907
1,8-GAAAAAAAAAAG,yta7,2,3,3,889,889,889,27668,0.003375,0.002250,0.001126,0.002700,0.000696,0.009830,0.000273,0.008103,0.003575,-2.359866
2,10-RGRGGARG,sua7,50,71,71,7340,2046,7340,2046,0.034702,0.006812,0.005356,0.011388,0.027200,0.043572,0.005060,0.008971,0.029518,-0.117975
3,9-TAGCCGCCGARG,toa1,5,9,9,1517,1119,1517,1119,0.008043,0.003296,0.001900,0.004676,0.003684,0.015213,0.001071,0.007675,0.006101,-0.727622
4,2-CADCARYADCADCAWYANCA,spt5,4,7,7,1260,1260,1260,6172,0.005556,0.003175,0.001590,0.004040,0.002236,0.011413,0.000866,0.008108,0.005067,-1.179937


In [30]:
best_jaccard = pd.read_parquet(
    run_dir / "full_best_assnt/best_assnt_metrics_jaccard_ds_DNA_rossi_chipexo.parquet"
)

best_precision = pd.read_parquet(
    run_dir / "full_best_assnt/best_assnt_metrics_precision_lb_ds_DNA_rossi_chipexo.parquet"
)

best_recall = pd.read_parquet(
    run_dir / "full_best_assnt/best_assnt_metrics_recall_lb_ds_DNA_rossi_chipexo.parquet"
)

display(best_jaccard.head())
display(best_precision.head())
display(best_recall.head())

,feature_idx,jaccard,name
0,NaN,0.0,tfa1
1,NaN,0.0,lys20
2,NaN,0.0,mss11
3,NaN,0.0,hst1
4,NaN,0.0,bdp1


,feature_idx,precision_lb,name
0,NaN,0.0,spt20
1,NaN,0.0,ino2
2,NaN,0.0,spp1
3,NaN,0.0,stb5
4,NaN,0.0,ssn8


,feature_idx,recall_lb,name
0,NaN,0.0,war1
1,NaN,0.0,god1
2,NaN,0.0,swi5
3,NaN,0.0,tea1
4,NaN,0.0,maf1


In [31]:
for name, df in {
    "jaccard": best_jaccard,
    "precision_lb": best_precision,
    "recall_lb": best_recall,
}.items():
    print(name, df.shape)
    display(df.head(10))

jaccard (167, 3)


,feature_idx,jaccard,name
0,NaN,0.0,tfa1
1,NaN,0.0,lys20
2,NaN,0.0,mss11
3,NaN,0.0,hst1
4,NaN,0.0,bdp1
5,NaN,0.0,rpb3
6,NaN,0.0,soh1
7,NaN,0.0,yrr1
8,NaN,0.0,tpk2
9,NaN,0.0,brf1


precision_lb (167, 3)


,feature_idx,precision_lb,name
0,NaN,0.0,spt20
1,NaN,0.0,ino2
2,NaN,0.0,spp1
3,NaN,0.0,stb5
4,NaN,0.0,ssn8
5,NaN,0.0,rpb7
6,NaN,0.0,sut1
7,NaN,0.0,hir1
8,NaN,0.0,hot1
9,NaN,0.0,tho1


recall_lb (167, 3)


,feature_idx,recall_lb,name
0,NaN,0.000000,war1
1,NaN,0.000000,god1
2,NaN,0.000000,swi5
3,NaN,0.000000,tea1
4,NaN,0.000000,maf1
5,NaN,0.000000,hhf1
6,NaN,0.000000,swi1
7,8-GAAAAAAAAAAG,0.003564,rna14
8,NaN,0.000000,rsc3
9,NaN,0.000000,bdf2


In [32]:
pred = pd.read_parquet(
    "/s/project/ml4rg_students/2026/project15/working/"
    "binding_bench_inputs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper_predictions.parquet"
)

pred.shape, pred.head(), pred["feature_idx"].nunique()

((110390, 10),
   chrom   start     end            feature_idx      score strand    gene_id  \
 0     I   70915   70916  1-GAAAAAAAAAAAAAAAAAA  10.752027      +    YAL038W   
 1    II    9031    9032  1-GAAAAAAAAAAAAAAAAAA  10.752027      +  YBL107W-A   
 2    II  145907  145908  1-GAAAAAAAAAAAAAAAAAA  10.752027      +  YBL039C-A   
 3    II  145907  145908  1-GAAAAAAAAAAAAAAAAAA  10.752027      +    YBL039C   
 4    II  145908  145909  1-GAAAAAAAAAAAAAAAAAA  10.752027      +    YBL038W   
 
                                        sequence_name  fimo_start  fimo_stop  
 0  YAL038W|species_file=_saccharomyces_cerevisiae...         121        139  
 1  YBL107W-A|species_file=_saccharomyces_cerevisi...         755        773  
 2  YBL039C-A|species_file=_saccharomyces_cerevisi...         115        133  
 3  YBL039C|species_file=_saccharomyces_cerevisiae...         812        830  
 4  YBL038W|species_file=_saccharomyces_cerevisiae...         713        731  ,
 10)

In [33]:
pred["chrom"].value_counts().head(20)

chrom
IV      12848
VII     10108
XV       9396
XII      8707
XIII     8358
XVI      8137
II       7040
XIV      6854
X        6539
XI       5951
V        5391
VIII     5258
IX       3878
III      3856
Mito     3452
VI       2630
I        1987
Name: count, dtype: int64

In [34]:
full[["feature_idx", "name", "jaccard", "precision_lb", "recall_lb"]].sort_values(
    "jaccard", ascending=False
).head(20)

,feature_idx,name,jaccard,precision_lb,recall_lb
188,9-TAGCCGCCGARG,ume6,0.285714,0.616141,0.386154
235,9-TAGCCGCCGARG,pho23,0.044776,0.048061,0.018038
131,9-TAGCCGCCGARG,rxt3,0.042254,0.070702,0.037746
257,9-TAGCCGCCGARG,ume1,0.037383,0.076875,0.046721
286,9-TAGCCGCCGARG,sin3,0.028571,0.017526,0.006800
13,9-TAGCCGCCGARG,rsc3,0.012987,0.008377,0.008377
32,1-GAAAAAAAAAAAAAAAAAA,sua7,0.008727,0.027596,0.014444
104,8-GAAAAAAAAAAG,rpc25,0.008086,0.001298,0.003321
200,9-TAGCCGCCGARG,cyc8,0.007491,0.010525,0.004066
120,7-SCGGGTAAY,ume6,0.006993,0.009827,0.003797


In [35]:
best_jaccard.sort_values("jaccard", ascending=False).head(20)

,feature_idx,jaccard,name
85,9-TAGCCGCCGARG,0.285714,ume6
118,1-GAAAAAAAAAAAAAAAAAA,0.008727,sua7
99,8-GAAAAAAAAAAG,0.008086,rpc25
142,6-ATCATCRA,0.006487,ssl2
15,7-SCGGGTAAY,0.005525,stb5
120,10-RGRGGARG,0.005024,tfb4
104,4-YRYTAYYRHY,0.004621,ssl1
82,5-AGATGATGAHGAMR,0.004472,spt6
159,2-CADCARYADCADCAWYANCA,0.002288,nrd1
92,3-ATATATATATATATATATAT,0.001145,rpo21


In [36]:
best_precision.sort_values("precision_lb", ascending=False).head(20)

,feature_idx,precision_lb,name
92,9-TAGCCGCCGARG,0.616141,ume6
161,3-ATATATATATATATATATAT,0.028747,sua7
66,10-RGRGGARG,0.019463,tfb4
36,6-ATCATCRA,0.019338,ssl2
19,2-CADCARYADCADCAWYANCA,0.016603,ume1
164,4-YRYTAYYRHY,0.012961,ssl1
11,5-AGATGATGAHGAMR,0.007883,rtf1
47,8-GAAAAAAAAAAG,0.007438,tfa1
105,1-GAAAAAAAAAAAAAAAAAA,0.006609,tfb3
56,7-SCGGGTAAY,0.005167,ref2


In [37]:
best_recall.sort_values("recall_lb", ascending=False).head(20)

,feature_idx,recall_lb,name
50,9-TAGCCGCCGARG,0.386154,ume6
44,1-GAAAAAAAAAAAAAAAAAA,0.014444,sua7
19,6-ATCATCRA,0.006683,ssl2
140,10-RGRGGARG,0.005057,tfb3
111,4-YRYTAYYRHY,0.004807,ssl1
84,5-AGATGATGAHGAMR,0.004278,spt6
68,7-SCGGGTAAY,0.003645,ref2
36,2-CADCARYADCADCAWYANCA,0.003593,tfb4
7,8-GAAAAAAAAAAG,0.003564,rna14
46,3-ATATATATATATATATATAT,0.000160,toa1


In [38]:
for name, df, score in [
    ("jaccard", best_jaccard, "jaccard"),
    ("precision", best_precision, "precision_lb"),
    ("recall", best_recall, "recall_lb"),
]:
    print(name)
    print("rows:", len(df))
    print("assigned:", df["feature_idx"].notna().sum())
    print("positive:", (df[score] > 0).sum())
    display(df.sort_values(score, ascending=False).head(10))

jaccard
rows: 167
assigned: 10
positive: 10


,feature_idx,jaccard,name
85,9-TAGCCGCCGARG,0.285714,ume6
118,1-GAAAAAAAAAAAAAAAAAA,0.008727,sua7
99,8-GAAAAAAAAAAG,0.008086,rpc25
142,6-ATCATCRA,0.006487,ssl2
15,7-SCGGGTAAY,0.005525,stb5
120,10-RGRGGARG,0.005024,tfb4
104,4-YRYTAYYRHY,0.004621,ssl1
82,5-AGATGATGAHGAMR,0.004472,spt6
159,2-CADCARYADCADCAWYANCA,0.002288,nrd1
92,3-ATATATATATATATATATAT,0.001145,rpo21


precision
rows: 167
assigned: 10
positive: 10


,feature_idx,precision_lb,name
92,9-TAGCCGCCGARG,0.616141,ume6
161,3-ATATATATATATATATATAT,0.028747,sua7
66,10-RGRGGARG,0.019463,tfb4
36,6-ATCATCRA,0.019338,ssl2
19,2-CADCARYADCADCAWYANCA,0.016603,ume1
164,4-YRYTAYYRHY,0.012961,ssl1
11,5-AGATGATGAHGAMR,0.007883,rtf1
47,8-GAAAAAAAAAAG,0.007438,tfa1
105,1-GAAAAAAAAAAAAAAAAAA,0.006609,tfb3
56,7-SCGGGTAAY,0.005167,ref2


recall
rows: 167
assigned: 10
positive: 10


,feature_idx,recall_lb,name
50,9-TAGCCGCCGARG,0.386154,ume6
44,1-GAAAAAAAAAAAAAAAAAA,0.014444,sua7
19,6-ATCATCRA,0.006683,ssl2
140,10-RGRGGARG,0.005057,tfb3
111,4-YRYTAYYRHY,0.004807,ssl1
84,5-AGATGATGAHGAMR,0.004278,spt6
68,7-SCGGGTAAY,0.003645,ref2
36,2-CADCARYADCADCAWYANCA,0.003593,tfb4
7,8-GAAAAAAAAAAG,0.003564,rna14
46,3-ATATATATATATATATATAT,0.000160,toa1


In [39]:
summary = (
    best_jaccard[["feature_idx", "name", "jaccard"]]
    .merge(best_precision[["feature_idx", "precision_lb"]], on="feature_idx", how="left")
    .merge(best_recall[["feature_idx", "recall_lb"]], on="feature_idx", how="left")
    .sort_values("jaccard", ascending=False)
)

display(summary)

,feature_idx,name,jaccard,precision_lb,recall_lb
2045869,9-TAGCCGCCGARG,ume6,0.285714,0.616141,0.386154
2760694,1-GAAAAAAAAAAAAAAAAAA,sua7,0.008727,0.006609,0.014444
2341659,8-GAAAAAAAAAAG,rpc25,0.008086,0.007438,0.003564
3302974,6-ATCATCRA,ssl2,0.006487,0.019338,0.006683
369735,7-SCGGGTAAY,stb5,0.005525,0.005167,0.003645
...,...,...,...,...,...
1289973,NaN,dig1,0.000000,0.000000,0.000000
1289974,NaN,dig1,0.000000,0.000000,0.000000
1289975,NaN,dig1,0.000000,0.000000,0.000000
1289976,NaN,dig1,0.000000,0.000000,0.000000


In [40]:
summary.to_csv(
    "/s/project/ml4rg_students/2026/project15/working/binding_bench_runs/"
    "fimo_streme__saccharomyces_cerevisiae_sequence_mapper/"
    "streme_binding_bench_summary.tsv",
    sep="\t",
    index=False,
)

STREME-FIMO recovers a strong UME6-like motif. The top motif
TAGCCGCCGARG is assigned to UME6 with high precision_lb and recall_lb,
whereas most other discovered motifs show weak or low-complexity matches.

In [41]:
from pathlib import Path
import re
import pandas as pd

input_dir = Path("/s/project/ml4rg_students/2026/project15/working/binding_bench_inputs")

pred_path = input_dir / "fimo_streme__saccharomyces_cerevisiae_sequence_mapper_predictions.parquet"
rank_path = input_dir / "fimo_streme__saccharomyces_cerevisiae_sequence_mapper_feature_ranks.parquet"

pred = pd.read_parquet(pred_path)
ranks = pd.read_parquet(rank_path)

def consensus(feature_idx):
    return str(feature_idx).split("-", 1)[1] if "-" in str(feature_idx) else str(feature_idx)

def is_low_complexity(feature_idx):
    seq = consensus(feature_idx).upper()

    # lange Homopolymere: AAAAAAAAA, TTTTTTTT, ...
    if re.search(r"([ACGT])\1{7,}", seq):
        return True

    # sehr AT-lastig und repetitiv
    at_fraction = sum(base in "AT" for base in seq) / max(len(seq), 1)
    if at_fraction >= 0.85 and len(seq) >= 10:
        return True

    # einfache Dinukleotid-Repeats wie ATATATATAT
    if re.fullmatch(r"([ACGT]{2})\1{4,}", seq):
        return True

    return False

features = pd.Series(pred["feature_idx"].unique())
low_complexity = set(features[features.map(is_low_complexity)])

low_complexity

{'1-GAAAAAAAAAAAAAAAAAA', '3-ATATATATATATATATATAT', '8-GAAAAAAAAAAG'}

In [42]:
filtered_pred = pred[~pred["feature_idx"].isin(low_complexity)].copy()
filtered_ranks = ranks[~ranks["feature_idx"].isin(low_complexity)].copy()

# Ranks neu durchnummerieren
filtered_ranks = filtered_ranks.sort_values("feature_rank").copy()
filtered_ranks["feature_rank"] = range(1, len(filtered_ranks) + 1)

filtered_pred_path = input_dir / "fimo_streme__saccharomyces_cerevisiae_sequence_mapper_predictions_no_low_complexity.parquet"
filtered_rank_path = input_dir / "fimo_streme__saccharomyces_cerevisiae_sequence_mapper_feature_ranks_no_low_complexity.parquet"

filtered_pred.to_parquet(filtered_pred_path, index=False)
filtered_ranks.to_parquet(filtered_rank_path, index=False)

filtered_pred.shape, filtered_ranks

((21361, 10),
               feature_idx  feature_rank  feature_score
 1  2-CADCARYADCADCAWYANCA             1           -2.0
 3            4-YRYTAYYRHY             2           -4.0
 4        5-AGATGATGAHGAMR             3           -5.0
 5              6-ATCATCRA             4           -6.0
 6             7-SCGGGTAAY             5           -7.0
 8          9-TAGCCGCCGARG             6           -9.0
 9             10-RGRGGARG             7          -10.0)

In [43]:
!binding_bench discrete \
  --input /s/project/ml4rg_students/2026/project15/working/binding_bench_inputs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper_predictions_no_low_complexity.parquet \
  --dataset DNA_rossi_chipexo \
  --output_dir /s/project/ml4rg_students/2026/project15/working/binding_bench_runs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper_no_low_complexity \
  --metric jaccard precision_lb recall_lb \
  --best_assignment \
  --ba_by_feature_rank \
  --feature_rank_path /s/project/ml4rg_students/2026/project15/working/binding_bench_inputs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper_feature_ranks_no_low_complexity.parquet \
  --feature_rank_name motif_score \
  --display_name "STREME-FIMO no low complexity" \
  --overwrite

CHANGED!
CHANGEDs again
Restricting to provided regions...
/data/nasif12/home_if12/s_kmill/binding_bench/src/binding_bench/utils/genome.py:713: UserWarning: Sortedness of columns cannot be checked when 'by' groups provided
  matched = query_sorted.join_asof(
Number of background windows in regions_df: 248665
Keeping sites: False
/opt/modules/i12g/anaconda/envs/ml4rg_project15/lib/python3.14/site-packages/pybedtools/bedtool.py:3756: UserWarning: Default names for filetype bed are:
['chrom', 'start', 'end', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts']
but file has 15 fields; you can supply custom names with the `names` kwarg
  warn(
/opt/modules/i12g/anaconda/envs/ml4rg_project15/lib/python3.14/site-packages/pybedtools/bedtool.py:3756: UserWarning: Default names for filetype bed are:
['chrom', 'start', 'end', 'name', 'score', 'strand', 'thickStart', 'thickEnd', 'itemRgb', 'blockCount', 'blockSizes', 'blockStarts']
but file has

In [45]:
from pathlib import Path
import pandas as pd

base = Path(
    "/s/project/ml4rg_students/2026/project15/working/"
    "binding_bench_runs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper/"
    "binding_bench/discrete/DNA_rossi_chipexo"
)

filtered = Path(
    "/s/project/ml4rg_students/2026/project15/working/"
    "binding_bench_runs/fimo_streme__saccharomyces_cerevisiae_sequence_mapper_no_low_complexity/"
    "binding_bench/discrete/DNA_rossi_chipexo"
)

def load_best(run_dir, metric):
    return pd.read_parquet(
        run_dir / "full_best_assnt" / f"best_assnt_metrics_{metric}_ds_DNA_rossi_chipexo.parquet"
    )

base_j = load_best(base, "jaccard")
filt_j = load_best(filtered, "jaccard")

base_p = load_best(base, "precision_lb")
filt_p = load_best(filtered, "precision_lb")

base_r = load_best(base, "recall_lb")
filt_r = load_best(filtered, "recall_lb")

In [46]:
display(base_j.sort_values("jaccard", ascending=False).head(10))
display(filt_j.sort_values("jaccard", ascending=False).head(10))

,feature_idx,jaccard,name
85,9-TAGCCGCCGARG,0.285714,ume6
118,1-GAAAAAAAAAAAAAAAAAA,0.008727,sua7
99,8-GAAAAAAAAAAG,0.008086,rpc25
142,6-ATCATCRA,0.006487,ssl2
15,7-SCGGGTAAY,0.005525,stb5
120,10-RGRGGARG,0.005024,tfb4
104,4-YRYTAYYRHY,0.004621,ssl1
82,5-AGATGATGAHGAMR,0.004472,spt6
159,2-CADCARYADCADCAWYANCA,0.002288,nrd1
92,3-ATATATATATATATATATAT,0.001145,rpo21


,feature_idx,jaccard,name
88,9-TAGCCGCCGARG,0.285714,ume6
63,4-YRYTAYYRHY,0.025641,rcs1
134,6-ATCATCRA,0.020408,smc1
80,7-SCGGGTAAY,0.005525,stb5
105,10-RGRGGARG,0.005024,tfb4
81,5-AGATGATGAHGAMR,0.004472,spt6
165,2-CADCARYADCADCAWYANCA,0.004386,sua7
7,NaN,0.000000,srb6
8,NaN,0.000000,ecm22
9,NaN,0.000000,lys20


In [48]:
def summarize(df, metric):
    return {
        "assigned": df["feature_idx"].notna().sum(),
        "positive": (df[metric] > 0).sum(),
        "best_score": df[metric].max(),
        "best_tf": df.loc[df[metric].idxmax(), "name"],
        "best_feature": df.loc[df[metric].idxmax(), "feature_idx"],
    }

summary = pd.DataFrame([
    {"run": "base", "metric": "jaccard", **summarize(base_j, "jaccard")},
    {"run": "filtered", "metric": "jaccard", **summarize(filt_j, "jaccard")},
    {"run": "base", "metric": "precision_lb", **summarize(base_p, "precision_lb")},
    {"run": "filtered", "metric": "precision_lb", **summarize(filt_p, "precision_lb")},
    {"run": "base", "metric": "recall_lb", **summarize(base_r, "recall_lb")},
    {"run": "filtered", "metric": "recall_lb", **summarize(filt_r, "recall_lb")},
])

display(summary)

,run,metric,assigned,positive,best_score,best_tf,best_feature
0,base,jaccard,10,10,0.285714,ume6,9-TAGCCGCCGARG
1,filtered,jaccard,7,7,0.285714,ume6,9-TAGCCGCCGARG
2,base,precision_lb,10,10,0.616141,ume6,9-TAGCCGCCGARG
3,filtered,precision_lb,7,7,0.616141,ume6,9-TAGCCGCCGARG
4,base,recall_lb,10,10,0.386154,ume6,9-TAGCCGCCGARG
5,filtered,recall_lb,7,7,0.386154,ume6,9-TAGCCGCCGARG


We used STREME+FIMO as an unsupervised motif-discovery baseline. STREME was run on the S. cerevisiae validation regions without access to TF labels or binding sites. The discovered motifs were scanned back over the regions with FIMO and evaluated with BindingBench against Rossi ChIP-exo sites. After filtering low-complexity motifs, the strongest assignment remained UME6, indicating that the baseline recovers at least one biologically meaningful TF motif.